# Train the rice-leaf disease classifier (MobileNetV2 → TFLite)

Produces `model.tflite` + `labels.txt` for `LeafHealthAnalyzerTFLite.kt` in the
[RicePlantHealthDetectionApplication](https://github.com/5amuel02/RicePlantHealthDetectionApplication)
Android app. Without these two files, the app automatically falls back to the
hand-engineered `LeafHealthAnalyzerV3` heuristic analyzer — this notebook is what
upgrades it to a real trained model.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Takes about 15 minutes end to end. The last cell downloads the two output files —
drop both into `app/src/main/assets/` in the Android project and rebuild.

**Dataset:** ["Rice Leaf Disease Image Samples"](https://data.mendeley.com/datasets/fwcj7stb8r/1)
by Prabira Kumar Sethy & Santi Kumari Barpanda, Mendeley Data, DOI
[10.17632/fwcj7stb8r.1](https://doi.org/10.17632/fwcj7stb8r.1), licensed
**CC BY 4.0**. 5,932 images across four classes: Bacterial blight, Blast, Brown
Spot, and Tungro. This notebook downloads a Kaggle mirror of the same dataset for
convenience — attribution is to the original Mendeley source above regardless of
which host serves the bytes.

**Note on "healthy":** this dataset only contains diseased-leaf photos, so the
model has no "healthy" class. `LeafHealthAnalyzerTFLite.kt` handles this
deliberately: a low top-class confidence is reported as "needs a closer look"
rather than a false "healthy" claim.

In [ ]:
!pip install -q kagglehub tensorflow==2.16.1

## 1. Download the dataset

Needs a free Kaggle API token: **kaggle.com → your profile → Settings → API →
"Create New Token"**, which downloads a `kaggle.json` file. Running the cell below
will prompt you to upload it.

In [ ]:
import os
import pathlib

from google.colab import files

if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    print("Upload the kaggle.json you downloaded from kaggle.com/settings:")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    for fname in uploaded:
        os.rename(fname, os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
import kagglehub

# Kaggle mirror of the Mendeley CC BY 4.0 dataset (see the markdown cell above for
# full attribution). If this exact slug ever moves, search "rice leaf disease" on
# kaggle.com/datasets for a dataset matching DOI 10.17632/fwcj7stb8r.1 (four
# classes: Bacterial blight, Blast, Brown Spot, Tungro) and swap the slug below —
# everything after this cell auto-detects the actual class folders it finds, so
# it isn't sensitive to the exact folder-naming style a different mirror might use.
KAGGLE_DATASET = "nirmalsankalana/rice-leaf-disease-image"

dataset_root = pathlib.Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("Downloaded to:", dataset_root)

# The download sometimes nests everything one level deeper — find the directory
# that actually contains class subfolders.
def find_class_root(root: pathlib.Path) -> pathlib.Path:
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for candidate in candidates:
        subdirs = [d for d in candidate.iterdir() if d.is_dir()]
        if len(subdirs) >= 2 and all(any(f.suffix.lower() in (".jpg", ".jpeg", ".png") for f in d.iterdir()) for d in subdirs):
            return candidate
    return root

data_dir = find_class_root(dataset_root)
class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
print(f"Found {len(class_dirs)} classes in {data_dir}:")
for d in class_dirs:
    n_images = sum(1 for f in d.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png"))
    print(f"  {d.name}: {n_images} images")

## 2. Build train/validation datasets

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)

# image_dataset_from_directory sorts class_names alphabetically by folder name —
# this exact order is what the model's output index means, and what labels.txt
# must be written in (done in the export cell below).
class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes (in label-index order):", class_names)

## 3. Preprocessing & augmentation

MobileNetV2 expects pixels scaled to `[-1, 1]`. Augmentation (flip/rotate/zoom) is
applied to the training set only, to reduce overfitting on a ~6k-image dataset.

**Important:** this scaling is *not* baked into the exported model graph — it's
done here as a dataset `.map()` step, and `LeafHealthAnalyzerTFLite.kt` replicates
the exact same `(pixel / 127.5) - 1` formula on the Android side. If you change
this preprocessing, update the Kotlin analyzer to match, or the model will silently
produce garbage predictions.

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    .cache().shuffle(1000).prefetch(AUTOTUNE)
)
val_ds = (
    val_ds
    .map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    .cache().prefetch(AUTOTUNE)
)

## 4. Build the model (MobileNetV2 transfer learning)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 5. Train (frozen base)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop],
)

## 6. Fine-tune (unfreeze the top of MobileNetV2)

A short, low-learning-rate pass over the later convolutional blocks — squeezes out
a few more points of accuracy without the frozen-base training's short 10 epochs
overfitting to ImageNet features that don't matter for rice leaves.

In [ ]:
base_model.trainable = True
FINE_TUNE_FROM = 100  # freeze everything before this layer index
for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop],
)

## 7. Evaluate

In [ ]:
loss, accuracy = model.evaluate(val_ds)
print(f"Validation accuracy: {accuracy * 100:.1f}%")

if accuracy < 0.80:
    print(
        "\n⚠️  Accuracy is lower than the ~95-97% these papers typically report for "
        "this dataset/architecture. Common causes: the downloaded dataset doesn't "
        "actually match the 4 expected classes (check the class list printed in "
        "section 1), or training stopped too early. Inspect before shipping this "
        "model."
    )

## 8. Export to TFLite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # dynamic-range quantization
tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)

# Written in class_names order — this is the exact index order the model's output
# array uses, and what LeafHealthAnalyzerTFLite.kt assumes when reading this file.
with open("labels.txt", "w") as f:
    f.write("\n".join(class_names))

size_kb = os.path.getsize("model.tflite") / 1024
print(f"model.tflite: {size_kb:.0f} KB")
print(f"labels.txt: {class_names}")

In [ ]:
from google.colab import files

files.download("model.tflite")
files.download("labels.txt")

print(
    "\nDone. Drop both downloaded files into "
    "app/src/main/assets/ in the Android project and rebuild."
)